# 0.- Montar Drive
## Acceder Archivos de Drive

In [ ]:
from google.colab import drive
drive.mount('Dataset Capsula')

Mounted at /content/drive


# 1.- Instalar entorno de Spark

##1.1.- Instalar jdk, pyspark, hadoop

In [59]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://downloads.apache.org/spark/spark-3.5.9/spark-3.5.9-bin-hadoop3.tgz
!tar xf spark-3.5.9-bin-hadoop3.tgz
# Instalar las bibliotecas de Python necesarias
!pip install -q pyspark==3.5.9 &> /dev/null
!pip install -q findspark &> /dev/null

##1.2.- Creación de variables de entorno

In [4]:
import os # libreria de manejo del sistema operativo
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.9-bin-hadoop3"
os.environ["PYTHON_PATH"] = "/content/spark-3.5.9-bin-hadoop3/python/lib/py4j-0.10.9-src.zip"

##1.3.- Creación SparkContext

In [5]:
import findspark
findspark.init()
from pyspark.sql import SparkSession
spark = SparkSession.builder\
        .master("local[*]")\
        .appName('Primer Ejercicio RDD')\
        .getOrCreate()
spark = SparkSession.builder.getOrCreate()
sc = spark.sparkContext
sc

<SparkContext master=local[*] appName=Primer Ejercicio RDD>

## 1.4.- Creación RDD desde arreglo,sin particionar





In [6]:
data = [1, 2, 3, 4, 5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30] #Creación arreglo data
rdd = sc.parallelize(data) # Sintaxis creación RDD
rdd.saveAsTextFile("./salida/rdd1") # Salvar RDD en archivo de texto en archivo de salida

In [7]:
# Ver el contenido
rdd.collect()

[1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 29,
 30]

In [8]:
rdd.take(3) #muestra hasta n elementos

[1, 2, 3]

In [9]:
rdd.reduce(lambda x,y: x+y) # realiza una reducción sobre el RDD utilizando la operación definida en el lambda (en este caso, la suma x + y).

465

In [10]:
rdd.reduce(lambda x,y: x*y) # 1*2 =2   2*3 = 6   6*4 = 24

265252859812191058636308480000000

In [11]:
rdd.top(5) #5 primeros descendente

[30, 29, 28, 27, 26]

In [12]:
rdd.first()

1

In [13]:
rdd.take(5)

[1, 2, 3, 4, 5]

In [14]:
rdd.count()

30

In [15]:
rdd.takeSample(False,6 , 2) #semilla

[4, 23, 18, 16, 1, 26]

In [16]:
rdd.sample(False,0.6).collect() #muestra colección aleatoria 50%

[1, 2, 3, 4, 6, 8, 9, 13, 14, 15, 19, 21, 24, 26, 30]

## 1.5.- Creación RDD desde arreglo, con particiones

In [17]:
data = [1, 2, 3, 4, 5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30] #Creación arreglo data
rdd1 = sc.parallelize(data,5) # Sintaxis creación RDD con cinco particiciones
rdd1.saveAsTextFile("./salida/rdd2") # Salvar RDD en archivo de texto en archivo de salida

In [18]:
# Ver el contenido de cada partición
particiones_contenido = rdd1.glom().collect()

# Imprimir el contenido de cada partición
for idx, particion in enumerate(particiones_contenido):
    print(f"Partición {idx}: {particion}")


Partición 0: [1, 2, 3, 4, 5, 6]
Partición 1: [7, 8, 9, 10, 11, 12]
Partición 2: [13, 14, 15, 16, 17, 18]
Partición 3: [19, 20, 21, 22, 23, 24]
Partición 4: [25, 26, 27, 28, 29, 30]


##1.6.- Creación RDD desde archivo de texto

In [21]:
rddT = sc.textFile('/content/drive/MyDrive/Dataset Capsula Semana 2/texto1.txt')
rddT.take(10)
##rddT.saveAsTextFile("./salida/rddT") # Salvar RDD en archivo de texto en archivo de salida


['Para que podemos usar el Big Data?',
 'Big Data es el enorme volumen de datos estructurados y no estructurados ',
 'con los que las empresas entran en contacto diariamente. ',
 'Las empresas de nuestro alrededor usan ',
 'Big Data ',
 'de muchas maneras. ',
 'Pero antes de entrar en la cuestion de para que se puede utilizar Big Data, ',
 'hablemos sobre qu� es ',
 'Big Data',
 'Segun la multinacional Oracle, Big Data tiene ']

In [22]:
rddF = rddT.filter(lambda x : 'Big' in x)
rddF.take(5)

['Para que podemos usar el Big Data?',
 'Big Data es el enorme volumen de datos estructurados y no estructurados ',
 'Big Data ',
 'Pero antes de entrar en la cuestion de para que se puede utilizar Big Data, ',
 'Big Data']

##1.7.- Creación RDD desde archivo de csv

In [23]:
rddTitanic = sc.textFile('/content/drive/MyDrive/Dataset Capsula Semana 2/Titanic.csv')
rddTitanic.take(20)
##rddTitanic.saveAsTextFile("./salida/Titanic") # Salvar RDD en archivo de texto en archivo de salida

['PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked',
 '1,0,3,"Braund, Mr. Owen Harris",male,22,1,0,A/5 21171,7.25,,S',
 '2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Thayer)",female,38,1,0,PC 17599,71.2833,C85,C',
 '3,1,3,"Heikkinen, Miss. Laina",female,26,0,0,STON/O2. 3101282,7.925,,S',
 '4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35,1,0,113803,53.1,C123,S',
 '5,0,3,"Allen, Mr. William Henry",male,35,0,0,373450,8.05,,S',
 '6,0,3,"Moran, Mr. James",male,,0,0,330877,8.4583,,Q',
 '7,0,1,"McCarthy, Mr. Timothy J",male,54,0,0,17463,51.8625,E46,S',
 '8,0,3,"Palsson, Master. Gosta Leonard",male,2,3,1,349909,21.075,,S',
 '9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27,0,2,347742,11.1333,,S',
 '10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14,1,0,237736,30.0708,,C',
 '11,1,3,"Sandstrom, Miss. Marguerite Rut",female,4,1,1,PP 9549,16.7,G6,S',
 '12,1,1,"Bonnell, Miss. Elizabeth",female,58,0,0,113783,26.55,C103,S',
 '

In [24]:
rddF = rddTitanic.filter(lambda x : 'female' in x)
rddF.take(5)

['2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Thayer)",female,38,1,0,PC 17599,71.2833,C85,C',
 '3,1,3,"Heikkinen, Miss. Laina",female,26,0,0,STON/O2. 3101282,7.925,,S',
 '4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35,1,0,113803,53.1,C123,S',
 '9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27,0,2,347742,11.1333,,S',
 '10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14,1,0,237736,30.0708,,C']

## 1.8.- Creación RDD desde archivo de json



In [25]:
rddJson = sc.textFile('/content/drive/MyDrive/Dataset Capsula Semana 2/ArchivoJson/*.json')
rddJson.take(5)
##rddJson.saveAsTextFile("./salida/Json") # Salvar RDD en archivo de texto en archivo de salida

['{"listing_id": "2977232", "reviewer_id": "23353304", "text": "Alesandra is very kind,powerful and nice person!! I\'ve never met to this great person!! If you have some trouble during your stay, she always   helps me whenever and wherever it is. Also her room is really comfortable and cute!!! It is easy to access from near station and convenient. I love this house :) Anyway thanks to her my NY trip was so great. If you\'re thinking of this room, i strongly recommend to you!!", "reviewer_name": "\\u590f\\u5b9f", "listing_longitude": "-73.94482017016313", "host_name": "Alesandra", "listing_name": "Low Winter Price by Central Park!", "date": "2015-01-02", "host_id": "15185319", "id": "24745582", "listing_latitude": "40.79285153420157"}',
 '{"listing_id": "2977232", "reviewer_id": "25012919", "text": "Alesandra was s really lovely host. She was always interested in me and gave me some nice tips. I enjoyed to stay in her apartment which is very inviting and comfortable. The subway is very 

# 2.- Práctica 1: Contar cantidad de palabras de un texto

## 2.1.- Leer Archivo de Texto

> Bloc con sangría



In [26]:
rdd2 = sc.textFile('/content/drive/MyDrive/Dataset Capsula Semana 2/texto1.txt')
rddT.take(12)

['Para que podemos usar el Big Data?',
 'Big Data es el enorme volumen de datos estructurados y no estructurados ',
 'con los que las empresas entran en contacto diariamente. ',
 'Las empresas de nuestro alrededor usan ',
 'Big Data ',
 'de muchas maneras. ',
 'Pero antes de entrar en la cuestion de para que se puede utilizar Big Data, ',
 'hablemos sobre qu� es ',
 'Big Data',
 'Segun la multinacional Oracle, Big Data tiene ',
 '"cuatro V": volumen, velocidad, variedad y valor:',
 'Volumen�es la gran cantidad de datos que se reciben.']

## 2.2.- Separar Pabalras del texto

In [27]:
# separa por palabras (split) con una palabra por registro
palabras = rdd2.flatMap(lambda x: x.split())
# Se imprimen las primeras 10 palabras
palabras.take(10)

['Para', 'que', 'podemos', 'usar', 'el', 'Big', 'Data?', 'Big', 'Data', 'es']

## 2.3.- Map:
###Mapea su entrada a alguna salida basada en la función especificada en la función.

In [28]:
# Genera las parejas <clave, valor> representandolas como la tupla (word, 1)
wc = palabras.map(lambda x: (x,1))
wc.take(12)

[('Para', 1),
 ('que', 1),
 ('podemos', 1),
 ('usar', 1),
 ('el', 1),
 ('Big', 1),
 ('Data?', 1),
 ('Big', 1),
 ('Data', 1),
 ('es', 1),
 ('el', 1),
 ('enorme', 1)]

## 2.4.- Reduce:
###Suma los valores para la misma clave. Spark internamente ordena por claves


In [29]:
# Suma los valores para la misma clave. Spark internamente ordena por claves
counts = wc.reduceByKey(lambda x,y:x+y)
counts.take(10)

[('usar', 2),
 ('Big', 8),
 ('es', 2),
 ('enorme', 1),
 ('volumen', 1),
 ('estructurados', 3),
 ('no', 2),
 ('los', 5),
 ('las', 1),
 ('empresas', 2)]

## 2.5.-SortByKey:
###Función que realiza la clasificación en un par (clave, valor) RDD basado en las claves. De forma predeterminada, la clasificación se realizará en orden ascendente. Para clasificar en orden descendente, pase “ascending=False”.

In [30]:
countsr = counts.map(lambda x: (x[1],x[0])).sortByKey(ascending=False)
#Print rdd6 result to console
countsr.take(10)

[(15, 'de'),
 (8, 'Big'),
 (7, 'que'),
 (7, 'datos'),
 (6, 'Data'),
 (5, 'los'),
 (5, 'la'),
 (4, 'el'),
 (3, 'estructurados'),
 (3, 'puede')]

## 2.6.- Sample
###Devuelve un subconjunto del RDD

In [31]:
rdd1= countsr.sample(False,0.5,1)
rdd1.take(50)


[(15, 'de'),
 (7, 'que'),
 (5, 'la'),
 (3, 'puede'),
 (3, 'valor'),
 (3, 'en'),
 (2, 'usar'),
 (2, 'gran'),
 (1, 'volumen'),
 (1, 'nuestro'),
 (1, 'cuestion'),
 (1, 'multinacional'),
 (1, 'Oracle,'),
 (1, '"cuatro'),
 (1, 'empresa'),
 (1, 'recibe.'),
 (1, 'estructurados?'),
 (1, 'sonido?'),
 (1, 'afectara'),
 (1, 'manera'),
 (1, 'por'),
 (1, 'derivarse'),
 (1, 'metodos.'),
 (1, 'diferentes'),
 (1, 'Data?'),
 (1, 'con'),
 (1, 'entran'),
 (1, 'alrededor'),
 (1, 'variedad'),
 (1, 'la�rapidez'),
 (1, 'Variedad�son'),
 (1, 'tipos'),
 (1, 'V�deo?'),
 (1, 'cambiando?'),
 (1, 'forma'),
 (1, 'son'),
 (1, 'capitalizar')]

##2.7.-saveAsTextFile
###Permite almacenar los datos en un archivo de salida

In [32]:
rdd1.saveAsTextFile("./salida/texto/output1")

##2.8. toDF
 Crea DataFrame desde RDD

In [33]:
df=counts.toDF()
df.show()

+-------------+---+
|           _1| _2|
+-------------+---+
|         usar|  2|
|          Big|  8|
|           es|  2|
|       enorme|  1|
|      volumen|  1|
|estructurados|  3|
|           no|  2|
|          los|  5|
|          las|  1|
|     empresas|  2|
|     contacto|  1|
|      nuestro|  1|
|       muchas|  1|
|     maneras.|  1|
|         Pero|  1|
|       entrar|  1|
|           la|  5|
|     cuestion|  1|
|         para|  1|
|        puede|  3|
+-------------+---+
only showing top 20 rows



In [34]:
columna=['Palabra','Cantidad']
df=counts.toDF(columna)
df.show()

+-------------+--------+
|      Palabra|Cantidad|
+-------------+--------+
|         usar|       2|
|          Big|       8|
|           es|       2|
|       enorme|       1|
|      volumen|       1|
|estructurados|       3|
|           no|       2|
|          los|       5|
|          las|       1|
|     empresas|       2|
|     contacto|       1|
|      nuestro|       1|
|       muchas|       1|
|     maneras.|       1|
|         Pero|       1|
|       entrar|       1|
|           la|       5|
|     cuestion|       1|
|         para|       1|
|        puede|       3|
+-------------+--------+
only showing top 20 rows



#3.- Práctica 2: Creación y uso DataFrame

##3.1.- Crear DataFrame a partir de un archivo CSV
###Podemos crear un DataFrame usando un archivo CSV y podemos especificar varias opciones como un separador, encabezado, esquema, inferSchema y varias otras opciones.

In [35]:
archivo = ('/content/drive/MyDrive/Dataset Capsula Semana 2/Titanic.csv')

df = spark.read.csv(archivo, sep=',', inferSchema=True, header=True)
# imprimir tipo de archivo
print(df)

DataFrame[PassengerId: int, Survived: int, Pclass: int, Name: string, Sex: string, Age: double, SibSp: int, Parch: int, Ticket: string, Fare: double, Cabin: string, Embarked: string]


In [36]:
df.show()

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|71.2833|  C85|       C|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|  7.925| NULL|       S|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|          113803|   53.1| C123|       S|
|          5|       0|     3|Allen, Mr. Willia...|  male|35.0|    0|    0|          373450|   8.05| NULL|       S|
|          6|       0|     3|    Moran, Mr. James|  male|NULL|    0|    0|      

##3.2.- Operaciones básicas en DataFrames

###3.2.1.- Count
Cuenta el número de filas de dataframe

In [37]:
df.count()

891

###3.2.2.- columns
Muestra las columnas del dataframe

In [38]:
df.columns

['PassengerId',
 'Survived',
 'Pclass',
 'Name',
 'Sex',
 'Age',
 'SibSp',
 'Parch',
 'Ticket',
 'Fare',
 'Cabin',
 'Embarked']

###3.2.3.- dtypes
Muestra el tipo de datos de las columnas dentro del DataFrame

In [39]:
df.dtypes

[('PassengerId', 'int'),
 ('Survived', 'int'),
 ('Pclass', 'int'),
 ('Name', 'string'),
 ('Sex', 'string'),
 ('Age', 'double'),
 ('SibSp', 'int'),
 ('Parch', 'int'),
 ('Ticket', 'string'),
 ('Fare', 'double'),
 ('Cabin', 'string'),
 ('Embarked', 'string')]

In [40]:
from pyspark.sql.functions import col


# Cambiar el tipo de varias columnas  Casteo
df = df.withColumn("Age", col("Age").cast("int"))

df.dtypes

[('PassengerId', 'int'),
 ('Survived', 'int'),
 ('Pclass', 'int'),
 ('Name', 'string'),
 ('Sex', 'string'),
 ('Age', 'int'),
 ('SibSp', 'int'),
 ('Parch', 'int'),
 ('Ticket', 'string'),
 ('Fare', 'double'),
 ('Cabin', 'string'),
 ('Embarked', 'string')]

In [41]:
df = df.withColumn("Age", col("Age").cast("int")) \
       .withColumn("Fare", col("Fare").cast("float")) \


df.dtypes

[('PassengerId', 'int'),
 ('Survived', 'int'),
 ('Pclass', 'int'),
 ('Name', 'string'),
 ('Sex', 'string'),
 ('Age', 'int'),
 ('SibSp', 'int'),
 ('Parch', 'int'),
 ('Ticket', 'string'),
 ('Fare', 'float'),
 ('Cabin', 'string'),
 ('Embarked', 'string')]

###3.2.4.- schema
Comprueba cómo Spark almacena el esquema del DataFrame

In [42]:
df.schema

StructType([StructField('PassengerId', IntegerType(), True), StructField('Survived', IntegerType(), True), StructField('Pclass', IntegerType(), True), StructField('Name', StringType(), True), StructField('Sex', StringType(), True), StructField('Age', IntegerType(), True), StructField('SibSp', IntegerType(), True), StructField('Parch', IntegerType(), True), StructField('Ticket', StringType(), True), StructField('Fare', FloatType(), True), StructField('Cabin', StringType(), True), StructField('Embarked', StringType(), True)])

###3.2.5.- printSchema
Muestra el esquema del dataFrame

In [43]:
df.printSchema()

root
 |-- PassengerId: integer (nullable = true)
 |-- Survived: integer (nullable = true)
 |-- Pclass: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- SibSp: integer (nullable = true)
 |-- Parch: integer (nullable = true)
 |-- Ticket: string (nullable = true)
 |-- Fare: float (nullable = true)
 |-- Cabin: string (nullable = true)
 |-- Embarked: string (nullable = true)



### 3.2.6.-select
Seleccione columnas del DataFrame

In [44]:
df.select("PassengerId").show()

+-----------+
|PassengerId|
+-----------+
|          1|
|          2|
|          3|
|          4|
|          5|
|          6|
|          7|
|          8|
|          9|
|         10|
|         11|
|         12|
|         13|
|         14|
|         15|
|         16|
|         17|
|         18|
|         19|
|         20|
+-----------+
only showing top 20 rows



In [45]:
df.select("PassengerId", "Sex").show()


+-----------+------+
|PassengerId|   Sex|
+-----------+------+
|          1|  male|
|          2|female|
|          3|female|
|          4|female|
|          5|  male|
|          6|  male|
|          7|  male|
|          8|  male|
|          9|female|
|         10|female|
|         11|female|
|         12|female|
|         13|  male|
|         14|  male|
|         15|female|
|         16|female|
|         17|  male|
|         18|  male|
|         19|female|
|         20|female|
+-----------+------+
only showing top 20 rows



###3.2.7. Show
Muestra columnas y contenido de un DataFrame

In [46]:
df.show(truncate=False)

+-----------+--------+------+-------------------------------------------------------+------+----+-----+-----+----------------+-------+-----+--------+
|PassengerId|Survived|Pclass|Name                                                   |Sex   |Age |SibSp|Parch|Ticket          |Fare   |Cabin|Embarked|
+-----------+--------+------+-------------------------------------------------------+------+----+-----+-----+----------------+-------+-----+--------+
|1          |0       |3     |Braund, Mr. Owen Harris                                |male  |22  |1    |0    |A/5 21171       |7.25   |NULL |S       |
|2          |1       |1     |Cumings, Mrs. John Bradley (Florence Briggs Thayer)    |female|38  |1    |0    |PC 17599        |71.2833|C85  |C       |
|3          |1       |3     |Heikkinen, Miss. Laina                                 |female|26  |0    |0    |STON/O2. 3101282|7.925  |NULL |S       |
|4          |1       |1     |Futrelle, Mrs. Jacques Heath (Lily May Peel)           |female|35  |1  

###3.2.8 Filter
Filtrar las filas según alguna condición.
Intentemos encontrar las filas con PassengerId >= 20.
Hay diferentes formas de especificar la condición.

In [47]:
df.filter(df["PassengerId"] >= 20).show()
df.filter(df.PassengerId >= 20).show()

+-----------+--------+------+--------------------+------+----+-----+-----+----------+--------+-----------+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|    Ticket|    Fare|      Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+----------+--------+-----------+--------+
|         20|       1|     3|Masselmani, Mrs. ...|female|NULL|    0|    0|      2649|   7.225|       NULL|       C|
|         21|       0|     2|Fynney, Mr. Joseph J|  male|  35|    0|    0|    239865|    26.0|       NULL|       S|
|         22|       1|     2|Beesley, Mr. Lawr...|  male|  34|    0|    0|    248698|    13.0|        D56|       S|
|         23|       1|     3|"McGowan, Miss. A...|female|  15|    0|    0|    330923|  8.0292|       NULL|       Q|
|         24|       1|     1|Sloper, Mr. Willi...|  male|  28|    0|    0|    113788|    35.5|         A6|       S|
|         25|       0|     3|Palsson, Miss. To...|female|   8|    3|    

###3.2.9 drop
Elimina una columna en particular

In [48]:
Nwdf = df.drop("Parch")
Nwdf.show(5)

+-----------+--------+------+--------------------+------+---+-----+----------------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex|Age|SibSp|          Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+---+-----+----------------+-------+-----+--------+
|          1|       0|     3|Braund, Mr. Owen ...|  male| 22|    1|       A/5 21171|   7.25| NULL|       S|
|          2|       1|     1|Cumings, Mrs. Joh...|female| 38|    1|        PC 17599|71.2833|  C85|       C|
|          3|       1|     3|Heikkinen, Miss. ...|female| 26|    0|STON/O2. 3101282|  7.925| NULL|       S|
|          4|       1|     1|Futrelle, Mrs. Ja...|female| 35|    1|          113803|   53.1| C123|       S|
|          5|       0|     3|Allen, Mr. Willia...|  male| 35|    0|          373450|   8.05| NULL|       S|
+-----------+--------+------+--------------------+------+---+-----+----------------+-------+-----+--------+
only showing top 5 rows



### 3.2.10 Impotar Librerias de Funciones Sql
Para utilizar funciones agrupadas se debe importar las librerias de Sql

---

In [49]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

### 3.2.11 Crear tabla Temporal para trabajar sql


---



In [50]:
Nwdf.createOrReplaceTempView("Tabla")

### 3.2.12 Crear consultas en el dataframe utilizando sql


---

In [51]:
spark.sql("select * from Tabla where Age >30 and Sex='female'").show()

+-----------+--------+------+--------------------+------+---+-----+------------+--------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex|Age|SibSp|      Ticket|    Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+---+-----+------------+--------+-----+--------+
|          2|       1|     1|Cumings, Mrs. Joh...|female| 38|    1|    PC 17599| 71.2833|  C85|       C|
|          4|       1|     1|Futrelle, Mrs. Ja...|female| 35|    1|      113803|    53.1| C123|       S|
|         12|       1|     1|Bonnell, Miss. El...|female| 58|    0|      113783|   26.55| C103|       S|
|         16|       1|     2|Hewlett, Mrs. (Ma...|female| 55|    0|      248706|    16.0| NULL|       S|
|         19|       0|     3|Vander Planke, Mr...|female| 31|    1|      345763|    18.0| NULL|       S|
|         26|       1|     3|Asplund, Mrs. Car...|female| 38|    1|      347077| 31.3875| NULL|       S|
|         41|       0|     3|Ahlin, Mrs. Johan...|femal

In [52]:
spark.sql("select Pclass , count(PassengerId)  from Tabla group by Pclass").show()

+------+------------------+
|Pclass|count(PassengerId)|
+------+------------------+
|     1|               216|
|     3|               491|
|     2|               184|
+------+------------------+



In [53]:
spark.sql("select sex, sum(Age),round(AVG(Age),2),min(Age),max(Age),count(Age) from Tabla group by sex ").show()

+------+--------+------------------+--------+--------+----------+
|   sex|sum(Age)|round(avg(Age), 2)|min(Age)|max(Age)|count(Age)|
+------+--------+------------------+--------+--------+----------+
|female|    7283|              27.9|       0|      63|       261|
|  male|   13908|              30.7|       0|      80|       453|
+------+--------+------------------+--------+--------+----------+



### 3.2.13 Aggregations
Podemos usar la función groupBy para agrupar los datos y luego usar la función "agg" para realizar la agregación de datos agrupados.

---

In [54]:
(df.groupBy("Sex")
  .agg(
      count("Age").alias("count"),
      sum("Age").alias("sum"),
      max("Age").alias("max"),
      min("Age").alias("min"),
      round(avg("Age"),2).alias("avg")
        ).show()
)

+------+-----+-----+---+---+----+
|   Sex|count|  sum|max|min| avg|
+------+-----+-----+---+---+----+
|female|  261| 7283| 63|  0|27.9|
|  male|  453|13908| 80|  0|30.7|
+------+-----+-----+---+---+----+



### 3.2.14 Sorting
##### Ordena los datos según el "Fare". De forma predeterminada, la clasificación se realizará en orden ascendente.
---

In [55]:
df.sort("Name").show(5)

+-----------+--------+------+--------------------+------+----+-----+-----+------+------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|Ticket|  Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+------+------+-----+--------+
|        147|       1|     3|"Andersson, Mr. A...|  male|  27|    0|    0|350043|7.7958| NULL|       S|
|        519|       1|     2|"Angle, Mrs. Will...|female|  36|    1|    0|226875|  26.0| NULL|       S|
|        291|       1|     1|"Barber, Miss. El...|female|  26|    0|    0| 19877| 78.85| NULL|       S|
|        625|       0|     3|"Bowen, Mr. David...|  male|  21|    0|    0| 54636|  16.1| NULL|       S|
|        508|       1|     1|"Bradley, Mr. Geo...|  male|NULL|    0|    0|111427| 26.55| NULL|       S|
+-----------+--------+------+--------------------+------+----+-----+-----+------+------+-----+--------+
only showing top 5 rows



In [56]:
df.sort(desc("Age")).show(5)

+-----------+--------+------+--------------------+----+---+-----+-----+--------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name| Sex|Age|SibSp|Parch|  Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+----+---+-----+-----+--------+-------+-----+--------+
|        631|       1|     1|Barkworth, Mr. Al...|male| 80|    0|    0|   27042|   30.0|  A23|       S|
|        852|       0|     3| Svensson, Mr. Johan|male| 74|    0|    0|  347060|  7.775| NULL|       S|
|        494|       0|     1|Artagaveytia, Mr....|male| 71|    0|    0|PC 17609|49.5042| NULL|       C|
|         97|       0|     1|Goldschmidt, Mr. ...|male| 71|    0|    0|PC 17754|34.6542|   A5|       C|
|        117|       0|     3|Connors, Mr. Patrick|male| 70|    0|    0|  370369|   7.75| NULL|       Q|
+-----------+--------+------+--------------------+----+---+-----+-----+--------+-------+-----+--------+
only showing top 5 rows



### 3.2.15 Columnas derivadas

---

In [57]:
df.withColumn("Meses", col("Age") * 12).show()

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+-----+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|Meses|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+-----+
|          1|       0|     3|Braund, Mr. Owen ...|  male|  22|    1|    0|       A/5 21171|   7.25| NULL|       S|  264|
|          2|       1|     1|Cumings, Mrs. Joh...|female|  38|    1|    0|        PC 17599|71.2833|  C85|       C|  456|
|          3|       1|     3|Heikkinen, Miss. ...|female|  26|    0|    0|STON/O2. 3101282|  7.925| NULL|       S|  312|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|  35|    1|    0|          113803|   53.1| C123|       S|  420|
|          5|       0|     3|Allen, Mr. Willia...|  male|  35|    0|    0|          373450|   8.05| NULL|       S|  420|
|          6|       0|     3|   

## 3.2.16 Guardar un DataFrame como un archivo CSV

---

In [58]:
df.write.csv("./archivo_salida", sep=',', header=True, mode="overwrite")